# NEC V2 Diagnostic Notebook

Muc tieu:
- Chan doan vi sao mo hinh thieu generalization.
- Dinh luong projection-dependence va distribution shift.
- Dua ra huong uu tien xu ly tiep theo.

Notebook nay khong train lai model nang; no dung du lieu train/val va ket qua da co trong `outputs/`.


In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path

BASE = Path('.').resolve()
DATA_DIR = BASE / 'data'
OUT_DIR = BASE / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

print('BASE:', BASE)
print('DATA_DIR exists:', DATA_DIR.exists())
print('OUT_DIR exists:', OUT_DIR.exists())


In [ ]:
def load_pair_csv(path):
    arr = np.loadtxt(path, delimiter=',', dtype=np.float64)
    if arr.ndim != 2 or arr.shape[0] != 2:
        raise ValueError(f'Expected 2-row csv at {path}, got shape={arr.shape}')
    t = arr[0]
    y = arr[1]
    idx = np.argsort(t)
    return t[idx], y[idx]


def safe_mean(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.mean(x)) if x.size else np.nan


def safe_std(x):
    x = np.asarray(x, dtype=np.float64)
    return float(np.std(x)) if x.size else np.nan


def nrmse_pct(y_true, rmse):
    den = np.mean(y_true) + 1e-9
    return float(100.0 * rmse / den)


def nmae_pct(y_true, mae):
    den = np.mean(y_true) + 1e-9
    return float(100.0 * mae / den)


In [ ]:
regions = ['bello', 'cam', 'iquitos', 'phi', 'sanjuan', 'vn']
rows = []
for region in regions:
    train_path = DATA_DIR / f'{region}_train.csv'
    val_path = DATA_DIR / f'{region}_val.csv'
    if not train_path.exists() or not val_path.exists():
        continue

    t_tr, y_tr = load_pair_csv(train_path)
    t_va, y_va = load_pair_csv(val_path)

    dy_tr = np.diff(y_tr)
    dy_va = np.diff(y_va)

    tail_w = min(10, len(dy_tr))
    dy_tail = dy_tr[-tail_w:] if tail_w > 0 else np.array([], dtype=np.float64)

    g_tr = dy_tr / (y_tr[:-1] + 1.0) if len(y_tr) > 1 else np.array([], dtype=np.float64)
    g_va = dy_va / (y_va[:-1] + 1.0) if len(y_va) > 1 else np.array([], dtype=np.float64)
    g_tail = g_tr[-tail_w:] if tail_w > 0 else np.array([], dtype=np.float64)

    dy_ratio = safe_mean(dy_va) / (safe_mean(dy_tail) + 1e-9)
    g_ratio = safe_mean(g_va) / (safe_mean(g_tail) + 1e-9)

    level_gap = (float(y_va[0]) - float(y_tr[-1])) / (float(y_tr[-1]) + 1e-9)

    shift_score = (
        abs(np.log(max(abs(dy_ratio), 1e-9)))
        + abs(np.log(max(abs(g_ratio), 1e-9)))
        + abs(level_gap)
    )

    rows.append({
        'region': region,
        'train_n': len(y_tr),
        'val_n': len(y_va),
        'train_last': float(y_tr[-1]),
        'val_first': float(y_va[0]),
        'val_mean': float(np.mean(y_va)),
        'tail_dy_mean': safe_mean(dy_tail),
        'val_dy_mean': safe_mean(dy_va),
        'dy_ratio_val_over_train_tail': float(dy_ratio),
        'tail_growth_mean': safe_mean(g_tail),
        'val_growth_mean': safe_mean(g_va),
        'growth_ratio_val_over_train_tail': float(g_ratio),
        'level_gap_frac': float(level_gap),
        'shift_score': float(shift_score),
        'tail_dy_volatility': safe_std(dy_tail) / (abs(safe_mean(dy_tail)) + 1e-9),
    })

shift_df = pd.DataFrame(rows).sort_values('shift_score', ascending=False).reset_index(drop=True)
shift_df


In [ ]:
shift_out = OUT_DIR / 'diagnostic_data_shift.csv'
shift_df.to_csv(shift_out, index=False)
print('saved:', shift_out)


In [ ]:
method_files = [
    OUT_DIR / 'bello_methods_eval.csv',
    OUT_DIR / 'phi_methods_eval.csv',
    OUT_DIR / 'vn_ca_bestfix_compare.csv',
]

method_rows = []
for f in method_files:
    if not f.exists():
        continue
    df = pd.read_csv(f)

    if 'idea' not in df.columns:
        continue

    region_guess = None
    name = f.name.lower()
    if name.startswith('bello'):
        region_guess = 'bello'
    elif name.startswith('phi'):
        region_guess = 'phi'
    elif name.startswith('vn'):
        region_guess = 'vn'

    if region_guess is None:
        continue

    y_val = load_pair_csv(DATA_DIR / f'{region_guess}_val.csv')[1]

    for _, r in df.iterrows():
        rmse = float(r.get('val_rmse_81_90', np.nan))
        mae = float(r.get('val_mae_81_90', np.nan))
        raw_rmse = float(r.get('val_rmse_raw_81_90', np.nan))

        pdi = (raw_rmse - rmse) / (raw_rmse + 1e-9)

        jump_ratio = float(r.get('max_forecast_jump', np.nan)) / (float(r.get('dy_cap', np.nan)) + 1e-9)
        growth_ratio = float(r.get('max_forecast_growth', np.nan)) / (float(r.get('growth_cap', np.nan)) + 1e-12)

        res_cols = [c for c in df.columns if c.startswith('res_f')]
        if res_cols:
            res_mean = float(np.mean([float(r[c]) for c in res_cols]))
        else:
            res_mean = np.nan

        method_rows.append({
            'region': region_guess,
            'idea': r['idea'],
            'train_rel_rmse': float(r.get('train_rel_rmse', np.nan)),
            'val_mae': mae,
            'val_rmse': rmse,
            'val_rmse_raw': raw_rmse,
            'val_nmae_pct': nmae_pct(y_val, mae),
            'val_nrmse_pct': nrmse_pct(y_val, rmse),
            'val_nrmse_raw_pct': nrmse_pct(y_val, raw_rmse),
            'projection_dependence_index': float(pdi),
            'jump_to_cap_ratio': float(jump_ratio),
            'growth_to_cap_ratio': float(growth_ratio),
            'residual_mean': res_mean,
            'best_epoch': float(r.get('best_epoch', np.nan)),
        })

methods_df = pd.DataFrame(method_rows)
methods_df = methods_df.sort_values(['region', 'val_rmse']).reset_index(drop=True)
methods_df


In [ ]:
methods_out = OUT_DIR / 'diagnostic_methods_summary.csv'
methods_df.to_csv(methods_out, index=False)
print('saved:', methods_out)


In [ ]:
if methods_df.empty:
    raise RuntimeError('No method evaluation files found in outputs/.')

best_rows = methods_df.loc[methods_df.groupby('region')['val_rmse'].idxmin()].copy()
best_rows = best_rows.sort_values('region').reset_index(drop=True)

rec_rows = []
for _, b in best_rows.iterrows():
    region = b['region']
    shift = shift_df[shift_df['region'] == region].iloc[0]

    issues = []
    actions = []

    if abs(shift['dy_ratio_val_over_train_tail']) > 1.5 or abs(shift['dy_ratio_val_over_train_tail']) < 0.67:
        issues.append('regime_shift_strong')
        actions.append('Tang trong so loss cho tail va dynamic trend, uu tien horizon curriculum dat sat val regime.')

    if b['projection_dependence_index'] > 0.40:
        issues.append('projection_dependent')
        actions.append('Giam phu thuoc projection: dua them raw-forecast consistency vao checkpoint score va objective.')

    gap = b['val_nrmse_pct'] - 100.0 * b['train_rel_rmse']
    if gap > 1.0:
        issues.append('fit_forecast_gap')
        actions.append('Can doi lai data vs phys vs rollout loss; tang regularization o forecast horizon.')

    if b['val_nrmse_pct'] < 3.0 and b['val_rmse'] > 1000:
        issues.append('absolute_scale_effect')
        actions.append('Theo doi metric chuan hoa (NRMSE/sMAPE) theo region, khong dung RMSE tuyet doi de so cheo region.')

    if b['jump_to_cap_ratio'] > 0.80:
        issues.append('cap_boundary_saturated')
        actions.append('Tinh chinh cap estimation theo regime (tail-adaptive cap) thay vi cap co dinh tu train tail.')

    if not actions:
        actions.append('Mo hinh dang on trong metric chuan hoa; tiep tuc fine-tune nhe loss weight va scheduler.')

    rec_rows.append({
        'region': region,
        'best_idea': b['idea'],
        'val_nrmse_pct': float(b['val_nrmse_pct']),
        'val_nrmse_raw_pct': float(b['val_nrmse_raw_pct']),
        'projection_dependence_index': float(b['projection_dependence_index']),
        'dy_ratio_val_over_train_tail': float(shift['dy_ratio_val_over_train_tail']),
        'growth_ratio_val_over_train_tail': float(shift['growth_ratio_val_over_train_tail']),
        'issues': ';'.join(issues) if issues else 'none_major',
        'priority_actions': ' | '.join(actions),
    })

rec_df = pd.DataFrame(rec_rows)
rec_df


In [ ]:
rec_out = OUT_DIR / 'diagnostic_region_recommendations.csv'
rec_df.to_csv(rec_out, index=False)
print('saved:', rec_out)


In [ ]:
print('=== QUICK SUMMARY ===')
print('\n[Data shift top]')
print(shift_df[['region', 'dy_ratio_val_over_train_tail', 'growth_ratio_val_over_train_tail', 'shift_score']].head(6).to_string(index=False))

print('\n[Best per region by projected RMSE]')
print(best_rows[['region', 'idea', 'val_rmse', 'val_nrmse_pct', 'val_rmse_raw', 'val_nrmse_raw_pct', 'projection_dependence_index']].to_string(index=False))

print('\n[Recommended direction]')
print(rec_df[['region', 'issues', 'priority_actions']].to_string(index=False))
